# Model 3: GRU Encoder-Decoder (~92M) for Legal Text Pretraining

**Before running on Kaggle:**
1. Settings -> Accelerator -> **GPU T4 x2** (or P100)
2. Settings -> **Internet on**
3. Add-ons -> Secrets -> add your Hugging Face token under the name **`HF`**
4. Use **Save Version -> Save & Run All (Commit)**. Outputs end up in `/kaggle/working/cs781_phase1/` (`prediction.json`, `submission.zip`, loss plot, model).

**Smoke test first:** `SMOKE_TEST = True` in the config cell (the default) runs the whole pipeline at tiny
scale (small tokenizer, 20 training steps, 64 test prompts) in about 10-20 minutes, writing to
`/kaggle/working/cs781_phase1_smoke/`. If it finishes and creates `submission.zip`, set
`SMOKE_TEST = False` and commit again for the real run.

## 1. Setup: imports, Hugging Face login, GPU check

In [ ]:
import os, json, math, time, random, zipfile, heapq
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import matplotlib.pyplot as plt

from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient
from datasets import Dataset
from transformers import PreTrainedTokenizer, TrainingArguments, Trainer, TrainerCallback

HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print("Hugging Face login successful.")

print("CUDA available:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

## 2. Configuration

In [ ]:
# Data
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILE = "Phase_1_Pretraining/Train/train_supreme_court.json"
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

# BPE tokenizer
BPE_VOCAB_SIZE = 50000
BPE_TRAIN_CHARS = 300_000_000

# Training
TRAIN_TOKENS = 200_000_000
VAL_TOKENS = 5_000_000
TRAIN_BLOCK_SIZE = 1024
ENC_LEN, DEC_LEN = 768, 256          # prompt / continuation split; must add up to TRAIN_BLOCK_SIZE
EPOCHS = 1
MAX_STEPS = -1                       # -1 = full epoch
EVAL_STEPS = 2000                    # each evaluation is slow for a step-by-step decoder
SAVE_STEPS = 1000
TIME_LIMIT_HOURS = 6.5               # training stops after this (Kaggle sessions end at 12 h)
SEED = 42
random.seed(SEED)
assert ENC_LEN + DEC_LEN == TRAIN_BLOCK_SIZE

# Generation
TARGET_GENERATION_WORDS = 128
MAX_GENERATION_TOKENS = 384
TEMPERATURE = 0.1
TOP_P = 0.95
GENERATION_BATCH_SIZE = 32
NUM_TEST_EXAMPLES = None             # None = whole test set

OUTPUT_DIR = "/kaggle/working/cs781_phase1"

# ---- SMOKE TEST: True = tiny end-to-end run (~10-20 min) to check everything works.
# ---- Set to False for the real training run.
SMOKE_TEST = True
if SMOKE_TEST:
    BPE_VOCAB_SIZE, BPE_TRAIN_CHARS = 2000, 2_000_000
    TRAIN_TOKENS, VAL_TOKENS = 300_000, 30_000
    MAX_STEPS, EVAL_STEPS, SAVE_STEPS = 20, 10, 10
    NUM_TEST_EXAMPLES = 64
    OUTPUT_DIR = "/kaggle/working/cs781_phase1_smoke"   # separate folder: smoke files never mix with the real run
    print("*** SMOKE TEST MODE: tiny run, predictions are NOT meant for submission ***")

# Output paths
MODEL_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "legal-gru-seq2seq-92m")
TOKENIZER_DIR = os.path.join(OUTPUT_DIR, "bpe_tokenizer")
BPE_PATH = os.path.join(TOKENIZER_DIR, "bpe_tokenizer.json")
PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")
LOSS_PLOT_PATH = os.path.join(OUTPUT_DIR, "training_validation_loss.png")
os.makedirs(TOKENIZER_DIR, exist_ok=True)
print("Configuration loaded.")

## 3. BPE Tokenizer

In [ ]:
_SHIFT = 20
_MASK = (1 << _SHIFT) - 1
_EOW = _MASK


class ByteLevelBPE:
    def __init__(self, vocab_size=5000):
        self.target_vocab_size = vocab_size
        self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self.merges = []
        self.symbol_to_id = {}
        self._ranks = None
        self._cache = {}

    def fit(self, sentences):
        word_freq = Counter()
        for s in sentences:
            word_freq.update(s.split())

        sym = {}
        key = {}
        for b in range(256):
            i = len(self.itos)
            self.symbol_to_id[(b,)] = i
            self.itos.append([b])
            sym[i] = (b,)
            key[i] = chr(b)
        sym[_EOW] = (256,)
        key[_EOW] = chr(256)

        words, freqs = [], []
        for w, f in word_freq.items():
            words.append([b + 4 for b in w.encode("utf-8")] + [_EOW])
            freqs.append(f)
        del word_freq

        pc = defaultdict(int)
        where = defaultdict(set)
        for wi, w in enumerate(words):
            f = freqs[wi]
            for x, y in zip(w, w[1:]):
                p = (x << _SHIFT) | y
                pc[p] += f
                where[p].add(wi)

        heap = [(-c, key[p >> _SHIFT], key[p & _MASK], p) for p, c in pc.items() if c > 1]
        heapq.heapify(heap)

        itos, merges, s2id = self.itos, self.merges, self.symbol_to_id
        target = self.target_vocab_size
        heappop, heappush = heapq.heappop, heapq.heappush

        while len(itos) < target:
            while heap:
                negc, _, _, best = heappop(heap)
                if pc.get(best, 0) == -negc:
                    break
            else:
                break
            if -negc <= 1:
                break

            a, b = best >> _SHIFT, best & _MASK
            new = len(itos)
            new_sym = sym[a] + sym[b]
            merges.append([list(sym[a]), list(sym[b])])
            s2id[new_sym] = new
            itos.append(list(new_sym))
            sym[new] = new_sym
            key[new] = key[a] + key[b]

            changed = set()
            for wi in where.pop(best, ()):
                w = words[wi]
                f = freqs[wi]
                i = 0
                while i < len(w) - 1:
                    if w[i] == a and w[i + 1] == b:
                        if i > 0:
                            l = w[i - 1]
                            p = (l << _SHIFT) | a; pc[p] -= f; changed.add(p)
                            p = (l << _SHIFT) | new; pc[p] += f; changed.add(p); where[p].add(wi)
                        if i + 2 < len(w):
                            r = w[i + 2]
                            p = (b << _SHIFT) | r; pc[p] -= f; changed.add(p)
                            p = (new << _SHIFT) | r; pc[p] += f; changed.add(p); where[p].add(wi)
                        pc[best] -= f
                        w[i:i + 2] = [new]
                    i += 1

            pc.pop(best, None)
            changed.discard(best)
            for p in changed:
                c = pc[p]
                if c > 1:
                    heappush(heap, (-c, key[p >> _SHIFT], key[p & _MASK], p))
                elif c <= 0:
                    del pc[p]

            if len(merges) % 500 == 0:
                print(f"Merges: {len(merges):,} | Vocab: {len(itos):,} | Best pair count: {-negc:,}")

        self.vocab_size = len(itos)
        self._ranks = None
        self._cache = {}
        print(f"BPE trained: {self.vocab_size:,} vocabulary entries, {len(merges):,} merges")
        return self

    def _build_ranks(self):
        def to_id(t):
            t = tuple(t)
            return _EOW if t == (256,) else self.symbol_to_id[t]
        if not self.symbol_to_id:
            for i, s in enumerate(self.itos):
                if isinstance(s, list):
                    self.symbol_to_id[tuple(s)] = i
        self._ranks = {}
        for l, r in self.merges:
            self._ranks[(to_id(l) << _SHIFT) | to_id(r)] = self.symbol_to_id[tuple(l) + tuple(r)]

    def tokenize_word(self, word):
        out = self._cache.get(word)
        if out is not None:
            return out
        if self._ranks is None:
            self._build_ranks()
        ranks = self._ranks
        w = [b + 4 for b in word.encode("utf-8")]
        w.append(_EOW)

        while len(w) > 1:
            best_new, best_p = None, None
            for x, y in zip(w, w[1:]):
                n = ranks.get((x << _SHIFT) | y)
                if n is not None and (best_new is None or n < best_new):
                    best_new, best_p = n, (x, y)
            if best_new is None:
                break
            a, b = best_p
            i, res = 0, []
            L = len(w)
            while i < L:
                if i < L - 1 and w[i] == a and w[i + 1] == b:
                    res.append(best_new); i += 2
                else:
                    res.append(w[i]); i += 1
            w = res

        out = [1 if t == _EOW else t for t in w]
        self._cache[word] = out
        return out

    def encode_sentence(self, sentence):
        ids = []
        tw = self.tokenize_word
        for word in sentence.split():
            ids.extend(tw(word))
        return ids

    def encode_corpus(self, sentences):
        return [self.encode_sentence(s) for s in sentences]

    def save(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"merges": self.merges, "itos": self.itos,
                       "vocab_size": self.vocab_size}, f, ensure_ascii=False)

In [ ]:
_EOW_MARK = "▁"


def _bytes_to_unicode():
    bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("¡"), ord("¬") + 1)) + list(range(ord("®"), ord("ÿ") + 1))
    cs = bs[:]
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    return dict(zip(bs, map(chr, cs)))


class LegalBPETokenizer(PreTrainedTokenizer):
    vocab_files_names = {"vocab_file": "bpe_tokenizer.json"}
    model_input_names = ["input_ids", "attention_mask"]

    def __init__(self, vocab_file, **kwargs):
        with open(vocab_file, "r", encoding="utf-8") as f:
            data = json.load(f)
        bpe = ByteLevelBPE()
        bpe.itos = data["itos"]
        bpe.merges = data["merges"]
        bpe.vocab_size = data["vocab_size"]
        bpe.symbol_to_id = {}
        bpe._build_ranks()
        self.bpe = bpe
        self.eow_id = len(bpe.itos)

        byte_enc = _bytes_to_unicode()
        self.byte_decoder = {c: b for b, c in byte_enc.items()}
        self.id_to_token = list(bpe.itos[:4])
        for sym in bpe.itos[4:]:
            self.id_to_token.append("".join(_EOW_MARK if b == 256 else byte_enc[b] for b in sym))
        self.id_to_token.append(_EOW_MARK)
        self.token_to_id = {t: i for i, t in enumerate(self.id_to_token)}

        kwargs.setdefault("pad_token", "<pad>")
        kwargs.setdefault("unk_token", "<unk>")
        kwargs.setdefault("bos_token", "<s>")
        kwargs.setdefault("eos_token", "</s>")
        kwargs.setdefault("clean_up_tokenization_spaces", False)
        super().__init__(**kwargs)

    def encode_fast(self, text):
        eow, tw = self.eow_id, self.bpe.tokenize_word
        ids = []
        for word in text.split():
            ids.extend(eow if t == 1 else t for t in tw(word))
        return ids

    @property
    def vocab_size(self):
        return len(self.id_to_token)

    def get_vocab(self):
        return dict(self.token_to_id)

    def _tokenize(self, text):
        return [self.id_to_token[i] for i in self.encode_fast(text)]

    def _convert_token_to_id(self, token):
        return self.token_to_id.get(token, self.unk_token_id)

    def _convert_id_to_token(self, index):
        return self.id_to_token[index]

    def convert_tokens_to_string(self, tokens):
        out = bytearray()
        for tok in tokens:
            if tok in ("<pad>", "<unk>", "<s>", "</s>"):
                continue
            for c in tok:
                out.extend(b" " if c == _EOW_MARK else bytes([self.byte_decoder[c]]))
        return out.decode("utf-8", errors="replace").strip()

    def save_vocabulary(self, save_directory, filename_prefix=None):
        path = os.path.join(save_directory, (filename_prefix + "-" if filename_prefix else "") + "bpe_tokenizer.json")
        self.bpe.save(path)
        return (path,)

## 4. Download Training Data

In [ ]:
train_file_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=TRAIN_FILE, repo_type="dataset", token=HF_TOKEN)
with open(train_file_path, "r", encoding="utf-8") as f:
    records = json.load(f)
print(f"Loaded {len(records):,} documents from {train_file_path}")

## 5. Train BPE Tokenizer (skipped if already trained)

In [ ]:
def bpe_training_texts(records, max_chars, seed=SEED):
    order = list(range(len(records)))
    random.Random(seed).shuffle(order)
    total = 0
    for idx in order:
        text = records[idx]["text"]
        yield text
        total += len(text)
        if total >= max_chars:
            break
    print(f"BPE training text: {total:,} characters")


if os.path.exists(BPE_PATH):
    print("Found existing BPE tokenizer, skipping training:", BPE_PATH)
else:
    start_time = time.time()
    bpe = ByteLevelBPE(vocab_size=BPE_VOCAB_SIZE)
    bpe.fit(bpe_training_texts(records, BPE_TRAIN_CHARS))
    bpe.save(BPE_PATH)
    print(f"Tokenizer training time: {(time.time() - start_time) / 60:.1f} min")
    print("Saved BPE tokenizer to:", BPE_PATH)

tokenizer = LegalBPETokenizer(vocab_file=BPE_PATH)
print("Vocabulary size:", len(tokenizer))

sample = " ".join(records[0]["text"].split()[:60])
sample_ids = tokenizer.encode_fast(sample)
print(f"{len(sample.split())} words -> {len(sample_ids)} tokens")
print("Round-trip OK:", tokenizer.decode(sample_ids) == sample)

## 6. Train / Validation Tokens and Blocks

In [ ]:
train_tokens = []
val_tokens = []

random.seed(SEED)

for data in records:
    text = data["text"]
    tokens = tokenizer.encode_fast(text) + [tokenizer.eos_token_id]

    if random.random() < 0.05:
        if len(val_tokens) < VAL_TOKENS:
            remaining = (VAL_TOKENS - len(val_tokens))
            val_tokens.extend(tokens[:remaining])
    else:
        if len(train_tokens) < TRAIN_TOKENS:
            remaining = (TRAIN_TOKENS - len(train_tokens))
            train_tokens.extend(tokens[:remaining])

    if (len(train_tokens) >= TRAIN_TOKENS and len(val_tokens) >= VAL_TOKENS):
        break

print(f"Train tokens: {len(train_tokens):,}")
print(f"Validation tokens: {len(val_tokens):,}")

In [ ]:
def create_blocks(tokens, block_size):
    total_length = (len(tokens) // block_size) * block_size
    tokens = tokens[:total_length]
    return [tokens[i:i + block_size] for i in range(0, total_length, block_size)]

train_dataset = Dataset.from_dict({"input_ids": create_blocks(train_tokens, TRAIN_BLOCK_SIZE)})
val_dataset = Dataset.from_dict({"input_ids": create_blocks(val_tokens, TRAIN_BLOCK_SIZE)})
del train_tokens, val_tokens          # free RAM
print(train_dataset)
print(val_dataset)

## 7. Model: GRU Encoder-Decoder with Bahdanau Attention

In [ ]:
# ===== A-2 GRU encoder-decoder with Bahdanau attention, adapted for capstone pretraining =====
# Same design as A-2 Q3 (2-layer biGRU encoder + bridge, Bahdanau attention, 2-layer GRU decoder
# with input feeding). Capstone changes are marked "CAPSTONE".

@dataclass
class GRUSeq2SeqConfig:
    vocab_size: int
    emb_dim: int = 512
    hidden: int = 768
    enc_layers: int = 2
    dec_layers: int = 2
    attn_dim: int = 256
    enc_len: int = 1536          # max prompt tokens
    dec_len: int = 512           # continuation tokens per training example (enc_len + dec_len = block size)
    dropout: float = 0.0
    bos_token_id: int = 2
    pad_token_id: int = 0


class BahdanauAttention(nn.Module):
    """A-2 additive attention: score = v_a^T tanh(W_a s + U_a h_j).
    CAPSTONE: U_a h_j is computed ONCE per sequence (precompute_keys) instead of at every
    decoder step. Same math, ~dec_len times less work for that term."""
    def __init__(self, dec_hidden, enc_hidden, attn_dim=128):
        super().__init__()
        self.W_a = nn.Linear(dec_hidden, attn_dim)
        self.U_a = nn.Linear(enc_hidden, attn_dim)
        self.v_a = nn.Linear(attn_dim, 1)

    def precompute_keys(self, enc_outputs):
        return self.U_a(enc_outputs)                                   # (B, S, A)

    def forward(self, dec_hidden, keys, enc_outputs, mask):
        scores = self.v_a(torch.tanh(self.W_a(dec_hidden).unsqueeze(1) + keys)).squeeze(-1)   # (B, S)
        scores = scores.masked_fill(mask == 0, -1e4)
        weights = torch.softmax(scores, dim=1)
        context = torch.bmm(weights.unsqueeze(1), enc_outputs).squeeze(1)                    # (B, 2H)
        return context, weights


class GRUSeq2SeqLM(nn.Module):
    def __init__(self, c: GRUSeq2SeqConfig):
        super().__init__()
        self.cfg = c
        H, E = c.hidden, c.emb_dim
        self.embedding = nn.Embedding(c.vocab_size, E)                 # CAPSTONE: one vocab, shared enc/dec/output
        self.enc_rnn = nn.GRU(E, H, num_layers=c.enc_layers, batch_first=True, bidirectional=True,
                              dropout=c.dropout if c.enc_layers > 1 else 0.0)
        self.bridge = nn.Linear(2 * H, H)
        self.attention = BahdanauAttention(H, 2 * H, c.attn_dim)
        self.cells = nn.ModuleList([nn.GRUCell(E + 2 * H if l == 0 else H, H) for l in range(c.dec_layers)])
        # CAPSTONE: A-2 used Linear(H + 2H -> vocab) (~115M params at a 50k vocab). Instead project to the
        # embedding size and reuse the embedding matrix as the output layer (weight tying).
        self.out_proj = nn.Linear(H + 2 * H, E)
        self.drop = nn.Dropout(c.dropout)

    def encode(self, src, src_mask):
        """src: (B, S) RIGHT-padded prompt ids; src_mask: (B, S) 1 = real token."""
        self.enc_rnn.flatten_parameters()                              # CAPSTONE: keep cuDNN weights contiguous (multi-GPU)
        lengths = src_mask.sum(1).clamp(min=1)
        x = self.drop(self.embedding(src))
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, h = self.enc_rnn(packed)                                  # padding is skipped entirely
        out, _ = pad_packed_sequence(out, batch_first=True, total_length=src.size(1))
        dec_h0 = torch.tanh(self.bridge(torch.cat([h[-2], h[-1]], dim=1)))   # A-2 bridge
        states = [dec_h0 for _ in range(self.cfg.dec_layers)]
        return out, self.attention.precompute_keys(out), states

    def step(self, tok, states, enc_out, keys, mask):
        """One decoder step (A-2 AttnDecoderStep). Returns logits (B, V) and new states."""
        x = self.drop(self.embedding(tok))
        context, _ = self.attention(states[-1], keys, enc_out, mask)
        inp = torch.cat([x, context], dim=1)
        new_states = []
        for l, cell in enumerate(self.cells):
            h = cell(inp, states[l])
            new_states.append(h)
            inp = self.drop(h)
        feat = self.out_proj(torch.cat([new_states[-1], context], dim=1))
        return F.linear(feat, self.embedding.weight), new_states


class GRUSeq2SeqForLM(nn.Module):
    """Trainer wrapper. Full teacher forcing (ratio 1.0): standard for LM pretraining."""
    def __init__(self, cfg: GRUSeq2SeqConfig):
        super().__init__()
        self.lm_config = cfg
        self.model = GRUSeq2SeqLM(cfg)

    def forward(self, input_ids, attention_mask=None, labels=None):
        m, c = self.model, self.lm_config
        if attention_mask is None:
            attention_mask = torch.ones_like(input_ids)
        enc_out, keys, states = m.encode(input_ids, attention_mask)
        dec_in = torch.cat([torch.full_like(labels[:, :1], c.bos_token_id), labels[:, :-1]], dim=1)
        logits = []
        for t in range(dec_in.size(1)):
            lg, states = m.step(dec_in[:, t], states, enc_out, keys, attention_mask)
            logits.append(lg)
        logits = torch.stack(logits, dim=1)
        loss = F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), labels.reshape(-1), ignore_index=-100)
        return {"loss": loss, "logits": logits}


class GRUPrefixCollator:
    """Block -> (prompt, continuation). Prompt length random in [min_prompt, enc_len], RIGHT-padded
    (the GRU encoder packs sequences, so padding is never read)."""
    def __init__(self, enc_len, dec_len, pad_id, min_prompt=64, seed=SEED):
        self.enc_len, self.dec_len, self.pad_id, self.min_prompt = enc_len, dec_len, pad_id, min_prompt
        self.rng = random.Random(seed)

    def __call__(self, features):
        ids = torch.tensor([f["input_ids"] for f in features], dtype=torch.long)
        assert ids.size(1) == self.enc_len + self.dec_len, "block size must equal enc_len + dec_len"
        B = ids.size(0)
        src = torch.full((B, self.enc_len), self.pad_id, dtype=torch.long)
        mask = torch.zeros((B, self.enc_len), dtype=torch.long)
        for i in range(B):
            p = self.rng.randint(self.min_prompt, self.enc_len)
            src[i, :p] = ids[i, self.enc_len - p: self.enc_len]        # the p tokens right before the target
            mask[i, :p] = 1
        return {"input_ids": src, "attention_mask": mask, "labels": ids[:, self.enc_len:].clone()}


def save_gru_model(model, out_dir):
    model = model.module if hasattr(model, "module") else model
    os.makedirs(out_dir, exist_ok=True)
    torch.save(model.model.state_dict(), os.path.join(out_dir, "gru_model.pt"))
    with open(os.path.join(out_dir, "gru_config.json"), "w") as f:
        json.dump(asdict(model.lm_config), f, indent=2)


def load_gru_model(out_dir, map_location="cpu"):
    with open(os.path.join(out_dir, "gru_config.json")) as f:
        cfg = GRUSeq2SeqConfig(**json.load(f))
    m = GRUSeq2SeqForLM(cfg)
    m.model.load_state_dict(torch.load(os.path.join(out_dir, "gru_model.pt"), map_location=map_location))
    return m


def _sample_top_p(logits, temperature, top_p):
    probs = torch.softmax(logits / temperature, dim=-1)
    sorted_probs, sorted_idx = torch.sort(probs, dim=-1, descending=True)
    cum = torch.cumsum(sorted_probs, dim=-1)
    sorted_probs[(cum - sorted_probs) > top_p] = 0.0          # keep the smallest set with mass >= top_p
    sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
    choice = torch.multinomial(sorted_probs, 1)
    return sorted_idx.gather(-1, choice).squeeze(-1)


def generate_batch_gru(prompts, model, tokenizer, device, target_words=128, check_every=16):
    m, c = model.model, model.lm_config
    eos_id, pad_id = tokenizer.eos_token_id, tokenizer.pad_token_id
    seqs = []
    for p in prompts:
        text = p if isinstance(p, str) else ("" if p is None else str(p))
        ids = tokenizer.encode_fast(text)[-c.enc_len:]
        seqs.append(ids if ids else [eos_id])
    L = max(len(s) for s in seqs)
    src = torch.tensor([s + [pad_id] * (L - len(s)) for s in seqs], dtype=torch.long, device=device)   # right-padded
    mask = torch.tensor([[1] * len(s) + [0] * (L - len(s)) for s in seqs], dtype=torch.long, device=device)
    tok = torch.full((len(seqs),), c.bos_token_id, dtype=torch.long, device=device)
    finished = torch.zeros(len(seqs), dtype=torch.bool, device=device)
    out_ids = []
    with torch.inference_mode():
        enc_out, keys, states = m.encode(src, mask)                    # prompt encoded once
        for step in range(MAX_GENERATION_TOKENS):                      # O(1) work per new token: RNN state
            logits, states = m.step(tok, states, enc_out, keys, mask)
            nxt = _sample_top_p(logits.float(), TEMPERATURE, TOP_P)
            nxt = torch.where(finished, torch.full_like(nxt, eos_id), nxt)
            out_ids.append(nxt)
            finished |= nxt == eos_id
            tok = nxt
            if finished.all():
                break
            if (step + 1) % check_every == 0:
                texts = tokenizer.batch_decode(torch.stack(out_ids, 1), skip_special_tokens=True)
                if all(f or len(t.split()) > target_words for f, t in zip(finished.tolist(), texts)):
                    break
    gen = torch.stack(out_ids, 1)
    return [" ".join(tokenizer.decode(gen[i], skip_special_tokens=True).split()[:target_words]) for i in range(len(seqs))]

In [ ]:
gru_config = GRUSeq2SeqConfig(
    vocab_size=len(tokenizer),
    emb_dim=768, hidden=1024,            # ~91.7M params
    enc_layers=2, dec_layers=2, attn_dim=256,
    enc_len=ENC_LEN, dec_len=DEC_LEN, dropout=0.0,
    bos_token_id=tokenizer.bos_token_id, pad_token_id=tokenizer.pad_token_id,
)
model = GRUSeq2SeqForLM(gru_config)

# each 1024-token block -> (prompt of 64..768 tokens, the 256 tokens that follow it)
data_collator = GRUPrefixCollator(ENC_LEN, DEC_LEN, tokenizer.pad_token_id, min_prompt=64)

print(f"Model size: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters")   # expect ~91.7M
b = data_collator([train_dataset[0], train_dataset[1]])
print({k: tuple(v.shape) for k, v in b.items()})   # expect input_ids (2, 768), attention_mask (2, 768), labels (2, 256)

## 8. Training

In [ ]:
training_args = TrainingArguments(
    output_dir=MODEL_OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    max_steps=MAX_STEPS,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=1e-3,
    weight_decay=0.1,
    warmup_steps=500,
    lr_scheduler_type="cosine",
    fp16=torch.cuda.is_available(),
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    logging_steps=100,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    prediction_loss_only=True,       # eval only needs the loss; avoids storing huge logits in memory
    seed=SEED,
    report_to="none",
)


class TimeLimit(TrainerCallback):
    """Safety net: stop training after a fixed wall-clock budget."""
    def __init__(self, seconds):
        self.seconds = seconds
    def on_train_begin(self, args, state, control, **kwargs):
        self.t0 = time.time()
    def on_step_end(self, args, state, control, **kwargs):
        if time.time() - self.t0 > self.seconds:
            control.should_training_stop = True


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset.select(range(min(256, len(val_dataset)))),   # small eval set (decoder is step-by-step)
    data_collator=data_collator,
    processing_class=tokenizer,
    callbacks=[TimeLimit(TIME_LIMIT_HOURS * 3600)],
)

start_time = time.time()
trainer.train()
training_time = time.time() - start_time

blocks_seen = trainer.state.global_step * training_args.train_batch_size * training_args.gradient_accumulation_steps
print(f"Steps: {trainer.state.global_step:,} | Training time: {training_time / 3600:.2f} hours")
print(f"Tokens seen: {blocks_seen * TRAIN_BLOCK_SIZE:,} | Continuation tokens trained on: {blocks_seen * DEC_LEN:,}")

## 9. Validation Perplexity and Loss Plot

In [ ]:
eval_loss = trainer.evaluate()["eval_loss"]
print(f"Validation loss: {eval_loss:.6f}")
print(f"Validation perplexity: {math.exp(eval_loss):.4f}")

logs = trainer.state.log_history
train_log = [(l["step"], l["loss"]) for l in logs if "loss" in l]
eval_log = [(l["step"], l["eval_loss"]) for l in logs if "eval_loss" in l]

plt.figure(figsize=(10, 5))
if train_log:
    plt.plot(*zip(*train_log), label="Training Loss")
if eval_log:
    plt.plot(*zip(*eval_log), label="Validation Loss")
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("Phase 1 Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(LOSS_PLOT_PATH, dpi=200)
plt.show()
print("Loss plot saved to:", LOSS_PLOT_PATH)

## 10. Save Final Model

In [ ]:
save_gru_model(trainer.model, MODEL_OUTPUT_DIR)
tokenizer.save_pretrained(MODEL_OUTPUT_DIR)
print("Final model saved to:", MODEL_OUTPUT_DIR)

## 11. Load Test Data and Final Model

In [ ]:
test_file_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=TEST_FILE, repo_type="dataset", token=HF_TOKEN)
with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)
if NUM_TEST_EXAMPLES:
    test_data = dict(list(test_data.items())[:NUM_TEST_EXAMPLES])
print("Number of test examples:", len(test_data))

generation_tokenizer = LegalBPETokenizer.from_pretrained(MODEL_OUTPUT_DIR)
generation_model = load_gru_model(MODEL_OUTPUT_DIR)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generation_model = generation_model.to(device).eval()
print("Device:", device)
print(f"Parameters: {sum(p.numel() for p in generation_model.parameters()) / 1e6:.2f}M")

## 12. Generate 128-Word Predictions for the Test Set

In [ ]:
predictions = {}
items = list(test_data.items())
total_examples = len(items)

for start in range(0, total_examples, GENERATION_BATCH_SIZE):
    batch = items[start:start + GENERATION_BATCH_SIZE]
    batch_predictions = generate_batch_gru(
        prompts=[record["text"] for _, record in batch],
        model=generation_model,
        tokenizer=generation_tokenizer,
        device=device,
        target_words=TARGET_GENERATION_WORDS,
    )
    for (example_id, _), prediction in zip(batch, batch_predictions):
        predictions[str(example_id)] = {"text": prediction}

    completed = start + len(batch)
    print(f"Completed {completed}/{total_examples} ({100 * completed / total_examples:.1f}%)")

## 13. Save prediction.json and CodaBench submission.zip

In [ ]:
with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTION_PATH, arcname="prediction.json")

print("prediction.json saved:", PREDICTION_PATH)
print("Codabench submission created:", SUBMISSION_ZIP)